# 1. Model Explainability: Permutation Feature Importance

In this notebook, we compare:
1. **Default Gini Feature Importance** (built into tree models).
2. **Permutation Importance** on unseen test data.

We deliberately inject a high-cardinality random noise feature to expose how default feature importance gets fooled, and how permutation importance catches it.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

# Set seed for reproducibility
np.random.seed(42)

# Generate a synthetic classification dataset:
# 5 informative features, 2 redundant features, 3 useless noise features
X_raw, y = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    weights=[0.85, 0.15], # Imbalanced (15% minority class)
    random_state=42
)

feature_names = [f"feature_{i+1}" for i in range(10)]
df = pd.DataFrame(X_raw, columns=feature_names)

# INJECT TRAP: A completely random high-cardinality noise feature
# This has ZERO true relationship with y
df['random_high_cardinality_noise'] = np.random.uniform(0, 100000, size=len(df))

X = df.copy()

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Dataset Shape: {X.shape}")
print(f"Features: {list(X.columns)}")
df.head()

Dataset Shape: (2000, 11)
Features: ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'random_high_cardinality_noise']


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,random_high_cardinality_noise
0,1.719421,-0.995037,1.347159,-0.105095,2.238050,0.120165,0.630093,0.341180,0.365803,-1.431276,37454.011885
1,1.628344,-0.526241,0.116202,1.399840,1.955209,0.610469,-0.102266,1.759813,0.484695,-1.058643,95071.430641
2,0.855182,1.525454,1.932957,-2.284388,-1.581202,1.583316,0.696688,-0.174947,-0.090593,0.141984,73199.394181
3,-0.007153,-0.589655,0.092564,0.956371,-1.073790,-0.219023,0.434362,-1.748067,-1.201759,1.585682,59865.848420
4,0.713820,-0.559466,-0.411422,-0.700216,2.681279,0.096479,0.373257,-0.193602,0.746925,-2.103692,15601.864044


---
## Part 1: Train Baseline Random Forest & Check Default Gini Importance

Notice how Random Forest's built-in `feature_importances_` evaluates the injected random noise column.

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Evaluate Baseline Performance
y_test_probs = rf.predict_proba(X_test)[:, 1]
baseline_auc = roc_auc_score(y_test, y_test_probs)
print(f"Baseline Test ROC-AUC: {baseline_auc:.4f}\n")

# Extract default Gini feature importances
df_gini = pd.DataFrame({
    'Feature': X.columns,
    'Gini_Importance': rf.feature_importances_
}).sort_values(by='Gini_Importance', ascending=False).reset_index(drop=True)

print("=== DEFAULT GINI FEATURE IMPORTANCE (TRAIN SET FIT) ===")
display(df_gini)

Baseline Test ROC-AUC: 0.9692

=== DEFAULT GINI FEATURE IMPORTANCE (TRAIN SET FIT) ===


,Feature,Gini_Importance
0,feature_4,0.213014
1,feature_6,0.156233
2,feature_1,0.136662
3,feature_9,0.107602
4,feature_2,0.097142
5,feature_10,0.072874
6,feature_5,0.072006
7,feature_8,0.045122
8,random_high_cardinality_noise,0.035418
9,feature_3,0.032809


---
## Part 2: Compute Permutation Feature Importance on Held-Out Test Data

Now we compute `permutation_importance` directly on `X_test` and `y_test`.
* `n_repeats=10`: Shuffles each column 10 times to compute mean drop and standard deviation.
* Metric: `roc_auc`.

In [3]:
# Run Permutation Importance on held-out test data
perm_result = permutation_importance(
    rf, 
    X_test, 
    y_test, 
    scoring='roc_auc', 
    n_repeats=10, 
    random_state=42, 
    n_jobs=-1
)

# Organize into a clean DataFrame
df_perm = pd.DataFrame({
    'Feature': X.columns,
    'Importance_Mean_Drop': perm_result.importances_mean,
    'Importance_Std': perm_result.importances_std
}).sort_values(by='Importance_Mean_Drop', ascending=False).reset_index(drop=True)

print("=== PERMUTATION IMPORTANCE (HELD-OUT TEST DATA) ===")
display(df_perm)

=== PERMUTATION IMPORTANCE (HELD-OUT TEST DATA) ===


,Feature,Importance_Mean_Drop,Importance_Std
0,feature_1,0.078682,0.006569
1,feature_4,0.067967,0.008271
2,feature_9,0.051375,0.006367
3,feature_6,0.037567,0.005953
4,feature_5,0.033643,0.007943
5,feature_2,0.018134,0.003309
6,feature_10,0.009046,0.003073
7,feature_8,0.002650,0.002464
8,random_high_cardinality_noise,0.000588,0.002354
9,feature_7,0.000546,0.000955


---
## Part 3: Comparison & Key Takeaways

Look at the ranking of `random_high_cardinality_noise`:
1. **Under Gini Importance:** The noise column ranks artificially high because continuous floating points provide many arbitrary split points to lower in-sample impurity.
2. **Under Permutation Importance:** Shuffling `random_high_cardinality_noise` on test data produces a `0.000` (or negative) drop in ROC-AUC, exposing it as completely useless for generalization.

### Some things which need to understand here:

#### 1. What is the core purpose of Module 09 (Model Explainability)?
- High-performing models like Random Forests and XGBoost are black boxes—they give predictions, but not the reasoning behind them.
- In the real world, you cannot deploy a black box blindly:
    - Spurious correlations: A model might achieve a 95% score using a fake shortcut (e.g., memorizing random IDs or hospital tags instead of real disease patterns).
    - Legal compliance: In banking and healthcare, laws require you to explain why a decision was made (e.g., why a loan was rejected).
- Module 09 teaches you how to audit, verify, and explain what your model actually learned

#### 2. What is Notebook 1 proving?
- It proves that the built-in feature importance of tree models lies to you, especially when data contains random noise or high-cardinality columns. Attribute naming 
`.feature_importance_` is that built-in thing.
- It introduces Permutation Importance as an honest, test-data-based way to find which features truly matter.

#### 3. What is .feature_importances_ (Gini / Impurity Importance)?
- It is a learned attribute generated automatically after you run .fit().
- How it works: It measures how much tree nodes split on a feature to separate classes during training.
- The values: All feature scores sum up to 1.0 (100%). Higher value means the trees chose that feature more often for splits.

#### 4. Why does .feature_importances_ lie?
- High-Cardinality Bias: Features with many unique values (like random numbers or IDs) give trees endless opportunities to make neat cuts. The tree greedily splits on them, inflating their importance score.
- Training-Data Memorization: It is calculated entirely on the training set. If the model memorizes pure noise, Gini importance rewards that noise as a "top feature."

#### 5. What is Permutation Importance, and how does it work?
Instead of looking at training splits, it measures how much your model's score crashes on the unseen test set when a feature is corrupted.
- Does it pass only one column to the model?
    - No. It passes the entire test table ($X\_test$) with all columns intact, but it takes one specific column and randomly shuffles its values across rows (like shuffling a deck of cards).
- What does shuffling do?
    - It completely breaks the relationship between that feature and the target while keeping every other feature untouched.
- The calculation:
    1. The function silently calculates your baseline test score on the clean data (e.g., ROC-AUC = 0.90).
    2. It shuffles Feature A and recalculates the score (e.g., ROC-AUC = 0.82).
    3. Score Drop: $0.90 - 0.82 = 0.08$.

#### 6. Why does permutation_importance need both X_test and y_test?
- It needs X_test to feed through the model to get predictions.
- It needs y_test (ground truth) to evaluate whether those predictions got worse after shuffling. Without y_test, it cannot calculate an evaluation metric like Accuracy or ROC-AUC.

#### 7. What do Importance_Mean_Drop and Importance_Std mean in the output?
- `Importance_Mean_Drop`: The average drop in your scoring metric across multiple shuffles.
    - If scoring is roc_auc, a mean drop of 0.078 means scrambling this feature dropped ROC-AUC by ~0.08 points.
    - A near-zero value (e.g., 0.0005) means the feature has no real impact.
    - A negative value (e.g., -0.0019) means shuffling the feature actually improved the model's test performance—the feature was actively confusing the model.
- Importance_Std: The standard deviation across repeats. It tells you if the drop is steady and reliable, or just random variation.

#### 8. What are n_repeats and n_jobs?
- n_repeats=10: Shuffling is random. Running it 10 times prevents a single lucky or unlucky shuffle from skewing the results, giving a reliable mean and standard deviation.
- n_jobs=-1: Uses all CPU cores to evaluate multiple feature shuffles simultaneously.
    - Inside RandomForestClassifier(n_jobs=-1): Speeds up training by building multiple trees at the same time.
    - Inside permutation_importance(..., n_jobs=-1): The model is already trained; it speeds up testing by evaluating different shuffled columns across different CPU cores.

#### 9. Methods vs. Attributes: What is the difference?
- Methods (Actions): Have parentheses (). They run code and perform an operation (e.g., .fit(), .predict(), .transform()).
- Attributes (Stored Data): Do not have parentheses. They store information or properties of an object (e.g., .shape, .columns).
- Learned Attributes (ending in _): Created by Scikit-Learn only after .fit() has completed (e.g., .feature_importances_, .classes_, .coef_).

#### 10. What do you do after getting the Permutation results?
You use it for Feature Selection / Model Pruning:
- 

#### 2. how are we finding this importance?
- The very first thing to know here is, tree models have some in-built attribute naming `.feature_importance_` which **measures how much each feature decreased node impurity (split cleanly) during training.** as tree models create splits to make decisions, this attribute shows some values which show how much that feature dropped that quality.
- Now the problem with that attribute method is, when it comes to give a reason for the prediction in practical world, it can't do that, it just tells 